# Overview of the Action-Conditioned (AC) Predictor

![ac_predictor_overview](../resources/v0_7/ac_predictor_overview.png)

The BioJEPA-AC model is designed to take a cell state and a perturbation and predict a latent representation of the perturbed cell state. We use the [Cell State Encoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_cell_state_encoder_v0_7.ipynb) to take the input cell state and generate a representation. We also use the [Action Composer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_action_composer_v0_7.ipynb) to generate a unified perturbation latent. The ACPredictor is a transformer-based module that takes these latents and generates a predicted representation of the perturbed cell state including an uncertainty estimate.

This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To this end, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. The ACPredictor takes three inputs: the cell state latent (output by the Cell State Encoder), target indices to predict, and the perturbation latent (output by the Action Composer). We'll mock all three using small dimensions that are easy to trace by hand.


*We'll use dimensions to match our other explainers but will generate simpler data to keep track of*

In [2]:
batch = 2
num_genes = 8
embed_dim = 6       
n_perts = 2        
action_dim = 5     
heads = 2          
mlp_ratio = 4.0


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
def mock_data(dim_1, dim_2, low=1, high=9):
    return torch.from_numpy(np.round(np.random.uniform(low, high, size=(dim_1, dim_2)), 0)).float()

**Cell State Latent (context_latents)** 

This is the representation of the control cell as output by the Cell State Encoder, representing the gene latent embeddings for the non-perturbed cell. As a reminder, this shape is $[B, \text{num\_genes}, \text{embed\_dim}]$. 

In [4]:
context_latents = mock_data(batch * num_genes, embed_dim).reshape(batch, num_genes, embed_dim)
context_latents.shape, context_latents

(torch.Size([2, 8, 6]),
 tensor([[[3., 2., 3., 5., 4., 5.],
          [3., 9., 7., 2., 4., 6.],
          [2., 9., 5., 7., 7., 4.],
          [4., 6., 7., 3., 3., 6.],
          [5., 2., 4., 3., 5., 8.],
          [2., 9., 4., 5., 5., 1.],
          [7., 9., 2., 2., 4., 7.],
          [1., 4., 1., 7., 4., 7.]],
 
         [[5., 9., 8., 6., 6., 9.],
          [3., 1., 2., 9., 9., 5.],
          [4., 7., 1., 7., 6., 3.],
          [6., 8., 2., 8., 4., 7.],
          [3., 5., 2., 5., 5., 4.],
          [7., 5., 7., 7., 4., 8.],
          [7., 8., 4., 5., 8., 6.],
          [5., 3., 9., 5., 7., 3.]]]))

**Target Indices (target_indices)** 

We've built our model so that each cell does not have to have every gene predicted. What is predicted is controlled via Target Indices. The predictor uses a learned embedding per gene to create query tokens in the attention to drive prediction only on what's desired. Currently though, our code is just predicting all genes so our example will follow suit. 

In [5]:
target_indices = torch.arange(num_genes).unsqueeze(0).expand(batch, -1)
target_indices.shape, target_indices

(torch.Size([2, 8]),
 tensor([[0, 1, 2, 3, 4, 5, 6, 7],
         [0, 1, 2, 3, 4, 5, 6, 7]]))

**Perturbation Embedding (action_latents)** 

This is the representation of the perturbations for the cell as output by the Action Composer.  As a reminder, this shape is $[B, N_\text{pert}, \text{action\_dim}]$. 

In [6]:
action_latents = mock_data(batch * n_perts, action_dim).reshape(batch, n_perts, action_dim)
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[4., 4., 5., 2., 6.],
          [1., 3., 7., 8., 6.]],
 
         [[4., 8., 9., 3., 4.],
          [7., 4., 1., 4., 5.]]]))

## Forward Pass

The goal of this model is to move the context_latent based on the action_latent to a new position in the latent representation of cell states. We do this by
1. Projecting the action latent to the same dimensions as our cell state
2. Using cross attention to merge the cell state and latent.
3. Using self attention and nonlinearity in the transformer to shift the cell state
4. Using two heads to project out the mean and deviation (logvar) for each gene's latent representation.

We'll first start by projecting the action latent into the same dimensions as the cell using an MLP-like layer. 

### Adapter Projection MLP

The action composer's output lives in a different dimensional space than the cell state latents so we need to bridge the gap to be able to give the model a chance to identify how the perturbation impacts the different channels of the cell state. Instead of a simple upward projection, we'll use a multilayer perceptron with two layers. The nonlinearity between the two layers allows the adapter to learn a nonlinear mapping between the two spaces.

#### Adapter - Linear Layer 1

The first linear layer does the projection from the action dimensions to the embedding dimensions. We'll do incremental weights so you'll see the impact of each upscaling. 

In [7]:
a_l1 = nn.Linear(action_dim, embed_dim)
vs, d = action_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.1*cols)  
a_l1.weight = nn.Parameter(pattern)
nn.init.zeros_(a_l1.bias)
a_l1.weight, a_l1.bias

(Parameter containing:
 tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
         [0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
         [0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [8]:
action_emb = a_l1(action_latents)
action_emb.shape, action_emb

(torch.Size([2, 2, 6]),
 tensor([[[ 2.1000,  4.2000,  6.3000,  8.4000, 10.5000, 12.6000],
          [ 2.5000,  5.0000,  7.5000, 10.0000, 12.5000, 15.0000]],
 
         [[ 2.8000,  5.6000,  8.4000, 11.2000, 14.0000, 16.8000],
          [ 2.1000,  4.2000,  6.3000,  8.4000, 10.5000, 12.6000]]],
        grad_fn=<ViewBackward0>))

#### RMSNorm
Our next layer is normalization. For our modern transformer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following: 
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training so removing the mean-centering operation preserves the critical variance-bounding effect. Also the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm many times, we'll create a class for it. In the class you can see that we split out the calculation, first limiting the precision of X, then computing $\frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}}$, and finally multiplying by the per-channel weights $\cdot \gamma$.

Since our linear layer produced values with an incremental pattern across channels, you'll see that RMSNorm will rescale them so that the RMS magnitude becomes 1 while preserving the relative differences between channels.

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [10]:
a_rms = RMSNorm(embed_dim)
a_rms.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [11]:
action_emb = a_rms(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 6]),
 tensor([[[0.2568, 0.5136, 0.7703, 1.0271, 1.2839, 1.5407],
          [0.2568, 0.5136, 0.7703, 1.0271, 1.2839, 1.5407]],
 
         [[0.2568, 0.5136, 0.7703, 1.0271, 1.2839, 1.5407],
          [0.2568, 0.5136, 0.7703, 1.0271, 1.2839, 1.5407]]],
        grad_fn=<MulBackward0>))

#### Adapter - GELU Non-linearity 

Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0. Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1. After RMSNorm, our values are centered around small magnitudes, so you'll see GELU introduce a meaningful non-linearity. Small positive values will be slightly reduced while larger values pass through nearly unchanged.

In [12]:
a_gelu = nn.GELU()

action_emb = a_gelu(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 6]),
 tensor([[[0.1544, 0.3575, 0.6004, 0.8708, 1.1560, 1.4456],
          [0.1544, 0.3575, 0.6004, 0.8708, 1.1560, 1.4456]],
 
         [[0.1544, 0.3575, 0.6004, 0.8708, 1.1560, 1.4456],
          [0.1544, 0.3575, 0.6004, 0.8708, 1.1560, 1.4456]]],
        grad_fn=<GeluBackward0>))

#### Adapter - Linear Layer 2

Now that we've done our nonlinearity, we'll do a final layer to let the model learn how the embedding dimensions relate to one another.

In [13]:
a_l2 = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 1.0).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 1*cols)  
a_l2.weight = nn.Parameter(pattern)
nn.init.zeros_(a_l2.bias)
a_l2.weight, a_l2.bias

(Parameter containing:
 tensor([[1., 1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3., 3.],
         [4., 4., 4., 4., 4., 4.],
         [5., 5., 5., 5., 5., 5.],
         [6., 6., 6., 6., 6., 6.]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [14]:
action_emb = a_l2(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 6]),
 tensor([[[ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088],
          [ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088]],
 
         [[ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088],
          [ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088]]],
        grad_fn=<ViewBackward0>))

### Target Gene Embedding Projection

Now we'll create the learnable target embedding tokens for each gene position. This will help ensure that if we only want a subset of genes predicted we can do it reliably. We call these embeddings queries sometimes since they act like "search requests" telling the model: for each gene, predict what its representation will look like after perturbation.

In [15]:
mask_queries = nn.Embedding(num_genes, embed_dim)
d, vs = num_genes, embed_dim
cols  = torch.arange(d).unsqueeze(1)
rows = torch.arange(vs).unsqueeze(0)
pattern = 1.0*(rows + cols)  
mask_queries.weight = nn.Parameter(pattern)
mask_queries.weight

Parameter containing:
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 1.,  2.,  3.,  4.,  5.,  6.],
        [ 2.,  3.,  4.,  5.,  6.,  7.],
        [ 3.,  4.,  5.,  6.,  7.,  8.],
        [ 4.,  5.,  6.,  7.,  8.,  9.],
        [ 5.,  6.,  7.,  8.,  9., 10.],
        [ 6.,  7.,  8.,  9., 10., 11.],
        [ 7.,  8.,  9., 10., 11., 12.]], requires_grad=True)

In [16]:
queries = mask_queries(target_indices)
queries

tensor([[[ 0.,  1.,  2.,  3.,  4.,  5.],
         [ 1.,  2.,  3.,  4.,  5.,  6.],
         [ 2.,  3.,  4.,  5.,  6.,  7.],
         [ 3.,  4.,  5.,  6.,  7.,  8.],
         [ 4.,  5.,  6.,  7.,  8.,  9.],
         [ 5.,  6.,  7.,  8.,  9., 10.],
         [ 6.,  7.,  8.,  9., 10., 11.],
         [ 7.,  8.,  9., 10., 11., 12.]],

        [[ 0.,  1.,  2.,  3.,  4.,  5.],
         [ 1.,  2.,  3.,  4.,  5.,  6.],
         [ 2.,  3.,  4.,  5.,  6.,  7.],
         [ 3.,  4.,  5.,  6.,  7.,  8.],
         [ 4.,  5.,  6.,  7.,  8.,  9.],
         [ 5.,  6.,  7.,  8.,  9., 10.],
         [ 6.,  7.,  8.,  9., 10., 11.],
         [ 7.,  8.,  9., 10., 11., 12.]]], grad_fn=<EmbeddingBackward0>)

#### Concatenate Cell State and Target

Now that we have our target embeddings identified, we'll concatenate them with the context latent into one representation.  This representation will double the token dimension (dim=1). This model will use attention and other layers to update the target based on the context latents. At the end we'll slice out the target and use that for our model output. 

*This is the same pattern used in masked token prediction: visible tokens (context) and mask tokens (queries) are concatenated, attention mixes information, and then the mask positions are read off*

In [17]:
sequence = torch.cat([context_latents, queries], dim=1)
sequence.shape, sequence

(torch.Size([2, 16, 6]),
 tensor([[[ 3.,  2.,  3.,  5.,  4.,  5.],
          [ 3.,  9.,  7.,  2.,  4.,  6.],
          [ 2.,  9.,  5.,  7.,  7.,  4.],
          [ 4.,  6.,  7.,  3.,  3.,  6.],
          [ 5.,  2.,  4.,  3.,  5.,  8.],
          [ 2.,  9.,  4.,  5.,  5.,  1.],
          [ 7.,  9.,  2.,  2.,  4.,  7.],
          [ 1.,  4.,  1.,  7.,  4.,  7.],
          [ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 1.,  2.,  3.,  4.,  5.,  6.],
          [ 2.,  3.,  4.,  5.,  6.,  7.],
          [ 3.,  4.,  5.,  6.,  7.,  8.],
          [ 4.,  5.,  6.,  7.,  8.,  9.],
          [ 5.,  6.,  7.,  8.,  9., 10.],
          [ 6.,  7.,  8.,  9., 10., 11.],
          [ 7.,  8.,  9., 10., 11., 12.]],
 
         [[ 5.,  9.,  8.,  6.,  6.,  9.],
          [ 3.,  1.,  2.,  9.,  9.,  5.],
          [ 4.,  7.,  1.,  7.,  6.,  3.],
          [ 6.,  8.,  2.,  8.,  4.,  7.],
          [ 3.,  5.,  2.,  5.,  5.,  4.],
          [ 7.,  5.,  7.,  7.,  4.,  8.],
          [ 7.,  8.,  4.,  5.,  8.,  6.],
      

### Multilayer Transformer Block

The transformer block for our predictor still outputs a latent representation but has a different structure from the cell state encoder transformer block. Since we're combining the perturbation and the context latents, this transformer will first use cross-attention and then use self attention on the output. The cross-attention allows the predictor to inject information from the perturbation action embedding (via cross-attention) on the cell state and then let that information propagate across all gene positions (via self-attention).

The predictor block consists of 6 layers:
1. RMSNorm 1
2. Multi-headed linear cross-attention (mechanism injection)
3. RMSNorm 2
4. Multi-headed gated linear self-attention (dynamics propagation)
5. RMSNorm 3
6. SwiGLU MLP

Residual connections provide gradient bypassing around each of the three sub-layers. This set of layers is repeated based on how many layers are configured.

#### RMSNorm 1

We'll start with our first normalization before the cross-attention layer. We'll only normalize the sequence data, not the perturbation, as that is the content that is updated. You'll see this pattern show up in the residual connection also later. 

In [18]:
x = sequence
x.shape, x

(torch.Size([2, 16, 6]),
 tensor([[[ 3.,  2.,  3.,  5.,  4.,  5.],
          [ 3.,  9.,  7.,  2.,  4.,  6.],
          [ 2.,  9.,  5.,  7.,  7.,  4.],
          [ 4.,  6.,  7.,  3.,  3.,  6.],
          [ 5.,  2.,  4.,  3.,  5.,  8.],
          [ 2.,  9.,  4.,  5.,  5.,  1.],
          [ 7.,  9.,  2.,  2.,  4.,  7.],
          [ 1.,  4.,  1.,  7.,  4.,  7.],
          [ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 1.,  2.,  3.,  4.,  5.,  6.],
          [ 2.,  3.,  4.,  5.,  6.,  7.],
          [ 3.,  4.,  5.,  6.,  7.,  8.],
          [ 4.,  5.,  6.,  7.,  8.,  9.],
          [ 5.,  6.,  7.,  8.,  9., 10.],
          [ 6.,  7.,  8.,  9., 10., 11.],
          [ 7.,  8.,  9., 10., 11., 12.]],
 
         [[ 5.,  9.,  8.,  6.,  6.,  9.],
          [ 3.,  1.,  2.,  9.,  9.,  5.],
          [ 4.,  7.,  1.,  7.,  6.,  3.],
          [ 6.,  8.,  2.,  8.,  4.,  7.],
          [ 3.,  5.,  2.,  5.,  5.,  4.],
          [ 7.,  5.,  7.,  7.,  4.,  8.],
          [ 7.,  8.,  4.,  5.,  8.,  6.],
      

In [19]:
rms1= RMSNorm(embed_dim)
rms1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [20]:
x_norm1 = rms1(x)
x_norm1.shape, x_norm1

(torch.Size([2, 16, 6]),
 tensor([[[0.7833, 0.5222, 0.7833, 1.3056, 1.0445, 1.3056],
          [0.5262, 1.5787, 1.2279, 0.3508, 0.7016, 1.0525],
          [0.3273, 1.4730, 0.8183, 1.1456, 1.1456, 0.6547],
          [0.7870, 1.1805, 1.3772, 0.5902, 0.5902, 1.1805],
          [1.0242, 0.4097, 0.8193, 0.6145, 1.0242, 1.6387],
          [0.3974, 1.7881, 0.7947, 0.9934, 0.9934, 0.1987],
          [1.2034, 1.5473, 0.3438, 0.3438, 0.6877, 1.2034],
          [0.2132, 0.8528, 0.2132, 1.4924, 0.8528, 1.4924],
          [0.0000, 0.3303, 0.6606, 0.9909, 1.3212, 1.6514],
          [0.2568, 0.5136, 0.7703, 1.0271, 1.2839, 1.5407],
          [0.4155, 0.6233, 0.8311, 1.0388, 1.2466, 1.4543],
          [0.5209, 0.6946, 0.8682, 1.0418, 1.2155, 1.3891],
          [0.5952, 0.7440, 0.8928, 1.0416, 1.1904, 1.3392],
          [0.6500, 0.7800, 0.9100, 1.0400, 1.1700, 1.3001],
          [0.6921, 0.8074, 0.9227, 1.0381, 1.1534, 1.2688],
          [0.7252, 0.8288, 0.9324, 1.0360, 1.1396, 1.2432]],
 
         [[0

#### Multi-Headed Linear Cross-Attention

Next we'll run our cross-attention to allow updates from the perturbation. In the cross-attention layer, the query comes from the gene sequence (context + target) and the key/value come from the MLP processed action embedding. The perturbation information gets injected into the gene representations since with this attention each gene position can look at the perturbation tokens and pull in relevant information. We apply the following calculation

$$\begin{aligned}
Q &= \text{elu}(x_\text{norm}\, W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(\text{action\_emb}\, W_k^\top + b_k) + 1.0 \\
V &= \text{action\_emb}\, W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q (K^\top V)}{Q (\sum_{j} K_j)^\top + \epsilon} \\
\\
y &= \text{Attn}\, W_c^\top + b_c
\end{aligned}$$


*Since the key and value are based on the perturbation, we do not add in gating like we do with self-attention. The model relies on the learned projections and the subsequent self-attention layer to regulate information flow.* 

In [21]:
B, T_q, C = x_norm1.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 16, 6, 3)

**Cross-attention** 

For cross-attention, the key/value input comes from the action embedding rather than the sequence itself. This builds a relationship between each gene position and the perturbation tokens. You'll notice that `kv_input` has a shape based on the perturbation batch size and n_perts and that the cross-attention allows the cell state to pull from all of the cell's perturbations. 

In [22]:
kv_input = action_emb
kv_input.shape, kv_input

(torch.Size([2, 2, 6]),
 tensor([[[ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088],
          [ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088]],
 
         [[ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088],
          [ 4.5848,  9.1696, 13.7544, 18.3392, 22.9240, 27.5088]]],
        grad_fn=<ViewBackward0>))

In [23]:
T_kv = kv_input.size(1)
T_kv

2

**Query** 

Now let's calculate our query. The query is the current position's "search request." Every layer and head issues queries that look for relationships with the key/value source. Our query first calculates a linear weight $Q=x_{norm}W^\top+b$, resulting in a vector of $[B,T_q,C]$ where $T_q$ is our gene sequence length. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T_q,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T_q,C_{Heads}]$.

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [24]:
q_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj.weight, -0.1)
nn.init.constant_(q_proj.bias, 0)
q_proj.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [25]:
q = q_proj(x_norm1).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 16, 3]),
 tensor([[[[-0.5745, -0.5745, -0.5745],
           [-0.5438, -0.5438, -0.5438],
           [-0.5565, -0.5565, -0.5565],
           [-0.5706, -0.5706, -0.5706],
           [-0.5531, -0.5531, -0.5531],
           [-0.5166, -0.5166, -0.5166],
           [-0.5330, -0.5330, -0.5330],
           [-0.5117, -0.5117, -0.5117],
           [-0.4954, -0.4954, -0.4954],
           [-0.5392, -0.5392, -0.5392],
           [-0.5610, -0.5610, -0.5610],
           [-0.5730, -0.5730, -0.5730],
           [-0.5803, -0.5803, -0.5803],
           [-0.5850, -0.5850, -0.5850],
           [-0.5882, -0.5882, -0.5882],
           [-0.5905, -0.5905, -0.5905]],
 
          [[-0.5745, -0.5745, -0.5745],
           [-0.5438, -0.5438, -0.5438],
           [-0.5565, -0.5565, -0.5565],
           [-0.5706, -0.5706, -0.5706],
           [-0.5531, -0.5531, -0.5531],
           [-0.5166, -0.5166, -0.5166],
           [-0.5330, -0.5330, -0.5330],
           [-0.5117, -0.5117, -0.5117],
         

In [26]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 16, 3]),
 tensor([[[[0.5630, 0.5630, 0.5630],
           [0.5806, 0.5806, 0.5806],
           [0.5732, 0.5732, 0.5732],
           [0.5652, 0.5652, 0.5652],
           [0.5752, 0.5752, 0.5752],
           [0.5966, 0.5966, 0.5966],
           [0.5869, 0.5869, 0.5869],
           [0.5995, 0.5995, 0.5995],
           [0.6093, 0.6093, 0.6093],
           [0.5832, 0.5832, 0.5832],
           [0.5707, 0.5707, 0.5707],
           [0.5638, 0.5638, 0.5638],
           [0.5597, 0.5597, 0.5597],
           [0.5571, 0.5571, 0.5571],
           [0.5553, 0.5553, 0.5553],
           [0.5540, 0.5540, 0.5540]],
 
          [[0.5630, 0.5630, 0.5630],
           [0.5806, 0.5806, 0.5806],
           [0.5732, 0.5732, 0.5732],
           [0.5652, 0.5652, 0.5652],
           [0.5752, 0.5752, 0.5752],
           [0.5966, 0.5966, 0.5966],
           [0.5869, 0.5869, 0.5869],
           [0.5995, 0.5995, 0.5995],
           [0.6093, 0.6093, 0.6093],
           [0.5832, 0.5832, 0.5832],
       

**Key** 

Next, we calculate the key. The key acts as a matching tag/address for each token in the key/value source. It is compared with the query to produce relevance scores. Since this is cross-attention, K is projected from the action embedding, allowing the model to build relationships between genes and perturbations.

We calculate a linear weight $K=\text{action\_emb}\cdot W^\top+b$, resulting in a vector of $[B,T_{kv},C]$ where $T_{kv}$ is the number of perturbation tokens. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T_{kv},C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T_{kv},C_{Heads}]$.

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [27]:
k_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj.weight, 0.2)
nn.init.constant_(k_proj.bias, 0)
k_proj.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [28]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 2, 3]),
 tensor([[[[19.2561, 19.2561, 19.2561],
           [19.2561, 19.2561, 19.2561]],
 
          [[19.2561, 19.2561, 19.2561],
           [19.2561, 19.2561, 19.2561]]],
 
 
         [[[19.2561, 19.2561, 19.2561],
           [19.2561, 19.2561, 19.2561]],
 
          [[19.2561, 19.2561, 19.2561],
           [19.2561, 19.2561, 19.2561]]]], grad_fn=<TransposeBackward0>))

In [29]:
k = F.elu(k) + 1.0
k

tensor([[[[20.2561, 20.2561, 20.2561],
          [20.2561, 20.2561, 20.2561]],

         [[20.2561, 20.2561, 20.2561],
          [20.2561, 20.2561, 20.2561]]],


        [[[20.2561, 20.2561, 20.2561],
          [20.2561, 20.2561, 20.2561]],

         [[20.2561, 20.2561, 20.2561],
          [20.2561, 20.2561, 20.2561]]]], grad_fn=<AddBackward0>)

**Value** 

Next, we calculate the value. The value is the payload you actually mix in once something matches. For cross-attention, it's a learned projection of the action embedding so the model can copy the right kind of perturbation information into the gene representations.

We calculate a linear weight $V=\text{action\_emb}\cdot W^\top+b$, resulting in a vector of $[B,T_{kv},C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T_{kv},C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T_{kv},C_{Heads}]$.

For V we do not use ELU+1 since the QK product will act on V.

In [30]:
v_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj.weight, 1.0)
nn.init.constant_(v_proj.bias, 0)
v_proj.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [31]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 2, 3]),
 tensor([[[[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806]],
 
          [[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806]]],
 
 
         [[[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806]],
 
          [[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806]]]], grad_fn=<TransposeBackward0>))

**Normalization denominator** 

In attention, when using softmax, attention values become probabilities and sum to 1. With linear attention, we need to manually do this normalization.

In softmax attention, the softmax inherently normalizes so weights sum to 1. Linear attention doesn't have that, so we need to create the denominator $z$. We'll do this by summing the tokens per embedding dimension resulting in a $[B,Heads,C_{Heads},1]$ dimension that we can then use in the denominator of our attention calculation

In [32]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[40.5123],
           [40.5123],
           [40.5123]],
 
          [[40.5123],
           [40.5123],
           [40.5123]]],
 
 
         [[[40.5123],
           [40.5123],
           [40.5123]],
 
          [[40.5123],
           [40.5123],
           [40.5123]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator** 

We're now ready to calculate the rest of the denominator. The denominator is the sum of attention weights for query across all keys. Each query gets its own normalizing constant, so queries attending to high-magnitude keys don't get inflated outputs. We add a final epsilon to ensure the denominator is not zero. The result from this is that we sum across the head dimensions creating a $[B,Heads,T,1]$ result

In [33]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 16, 1]),
 tensor([[[[0.0146],
           [0.0142],
           [0.0144],
           [0.0146],
           [0.0143],
           [0.0138],
           [0.0140],
           [0.0137],
           [0.0135],
           [0.0141],
           [0.0144],
           [0.0146],
           [0.0147],
           [0.0148],
           [0.0148],
           [0.0149]],
 
          [[0.0146],
           [0.0142],
           [0.0144],
           [0.0146],
           [0.0143],
           [0.0138],
           [0.0140],
           [0.0137],
           [0.0135],
           [0.0141],
           [0.0144],
           [0.0146],
           [0.0147],
           [0.0148],
           [0.0148],
           [0.0149]]],
 
 
         [[[0.0148],
           [0.0136],
           [0.0142],
           [0.0144],
           [0.0146],
           [0.0148],
           [0.0148],
           [0.0144],
           [0.0135],
           [0.0141],
           [0.0144],
           [0.0146],
           [0.0147],
           [0.0148

**Numerator** 

Now we're ready to complete our numerator. This is just a matter of multiplying Q, K, and V. We'll need to transpose K to get the interaction between the query and key to then multiply against the value. 

Since we're looking to save memory, we actually first multiply the key and value to create head dimension matrixes, and then multiply by the query. This order of operations is part of what saves memory. 

In [34]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[3900.5457, 3900.5457, 3900.5457],
           [3900.5457, 3900.5457, 3900.5457],
           [3900.5457, 3900.5457, 3900.5457]],
 
          [[3900.5457, 3900.5457, 3900.5457],
           [3900.5459, 3900.5459, 3900.5459],
           [3900.5459, 3900.5459, 3900.5459]]],
 
 
         [[[3900.5454, 3900.5454, 3900.5454],
           [3900.5454, 3900.5454, 3900.5454],
           [3900.5454, 3900.5454, 3900.5454]],
 
          [[3900.5454, 3900.5454, 3900.5454],
           [3900.5454, 3900.5452, 3900.5452],
           [3900.5454, 3900.5452, 3900.5452]]]], grad_fn=<UnsafeViewBackward0>))

In [35]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 16, 3]),
 tensor([[[[6588.1494, 6588.1494, 6588.1494],
           [6793.4087, 6793.4087, 6793.4087],
           [6707.8140, 6707.8140, 6707.8140],
           [6613.8120, 6613.8120, 6613.8120],
           [6730.6377, 6730.6377, 6730.6377],
           [6980.7832, 6980.7832, 6980.7832],
           [6867.3296, 6867.3296, 6867.3296],
           [7014.9741, 7014.9741, 7014.9741],
           [7129.8853, 7129.8853, 7129.8853],
           [6824.3599, 6824.3599, 6824.3599],
           [6677.6665, 6677.6665, 6677.6665],
           [6597.6768, 6597.6768, 6597.6768],
           [6549.7354, 6549.7354, 6549.7354],
           [6518.8916, 6518.8916, 6518.8916],
           [6497.9365, 6497.9365, 6497.9365],
           [6483.0767, 6483.0767, 6483.0767]],
 
          [[6588.1494, 6588.1494, 6588.1494],
           [6793.4087, 6793.4087, 6793.4087],
           [6707.8140, 6707.8140, 6707.8140],
           [6613.8120, 6613.8120, 6613.8120],
           [6730.6387, 6730.6387, 6730.6387],
   

**Linear Attention** 

Now we're ready to normalize our numerator by the denominator. You'll see that because of our extremely consistent values and initialization we're ending up with very consistent values by example even across heads. 

In [36]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 16, 3]),
 tensor([[[[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806]],
 
          [[96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
           [96.2806, 96.2806, 96.2806],
         

**Collapse heads** 

We now can bring our heads back together. We have to undo our head splitting. First we flip our heads and tokens (genes) so that we have a $[B,T,Heads,C_{Heads}]$ and then we collapse the head and head embeddings to end up with $[B,T,C]$

In [37]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 16, 6]),
 tensor([[[96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2806, 96.2806],
          [96.2806, 96.2806, 96.2806, 96.2806, 96.2

**Cross-head final projection** 

Finally, we project the attention output through a final linear layer. This allows the model to learn how to combine information across the different heads.

*Notice that unlike self-attention, we skip the sigmoid gating step here. Cross-attention doesn't apply gating because the model regulates information flow through the learned Q/K/V projections and the downstream self-attention layer's gate.*

In [38]:
c_proj = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj.weight = nn.Parameter(pattern)
c_proj.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [39]:
x_attn_cross = c_proj(y)
x_attn_cross.shape, x_attn_cross

(torch.Size([2, 16, 6]),
 tensor([[[57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3815, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7542, 86.7547],
          [57.3814, 63.6252, 69.4296, 74.8119, 80.7

#### Residual Connection

We now have our linear cross-attention calculated and will use a residual connection to allow gradients to bypass the cross-attention layer. You'll see how much larger the residual connection impact is on our output compared to the cross-attention contribution.

For this residual connection, we only include the cell-state representation `x` and we do not create a residual connection to the action_latents. This is because the action_latents are the keys and values in cross-attention, not the queries. This means that they're read-only inputs and the cross-attention only updates the cell state input. Since the residual is meant to add a bypass path for backprop gradients, we only need to do it on paths that are changing, in this case the cell state latent. 

In [40]:
x = x + x_attn_cross
x.shape, x

(torch.Size([2, 16, 6]),
 tensor([[[60.3814, 65.6252, 72.4296, 79.8119, 84.7542, 91.7547],
          [60.3814, 72.6252, 76.4296, 76.8119, 84.7542, 92.7547],
          [59.3814, 72.6252, 74.4296, 81.8119, 87.7542, 90.7547],
          [61.3814, 69.6252, 76.4296, 77.8119, 83.7542, 92.7547],
          [62.3814, 65.6252, 73.4296, 77.8119, 85.7542, 94.7547],
          [59.3814, 72.6252, 73.4296, 79.8119, 85.7542, 87.7547],
          [64.3814, 72.6252, 71.4296, 76.8119, 84.7542, 93.7547],
          [58.3814, 67.6252, 70.4296, 81.8119, 84.7542, 93.7547],
          [57.3815, 64.6252, 71.4296, 77.8119, 84.7542, 91.7547],
          [58.3814, 65.6252, 72.4296, 78.8119, 85.7542, 92.7547],
          [59.3814, 66.6252, 73.4296, 79.8119, 86.7542, 93.7547],
          [60.3814, 67.6252, 74.4296, 80.8119, 87.7542, 94.7547],
          [61.3814, 68.6252, 75.4296, 81.8119, 88.7542, 95.7547],
          [62.3814, 69.6252, 76.4296, 82.8119, 89.7542, 96.7547],
          [63.3814, 70.6252, 77.4296, 83.8119, 90.7

#### RMSNorm 2

Now that we've calculated the cross-attention, we'll do another round of normalization before the self-attention layer. We'll use RMSNorm again.

In [41]:
rms2= RMSNorm(embed_dim)
rms2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [42]:
x_norm2 = rms2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 16, 6]),
 tensor([[[0.7887, 0.8571, 0.9460, 1.0424, 1.1070, 1.1984],
          [0.7747, 0.9318, 0.9806, 0.9855, 1.0874, 1.1901],
          [0.7565, 0.9252, 0.9482, 1.0422, 1.1179, 1.1562],
          [0.7910, 0.8972, 0.9849, 1.0027, 1.0793, 1.1953],
          [0.8056, 0.8475, 0.9483, 1.0049, 1.1074, 1.2237],
          [0.7707, 0.9426, 0.9531, 1.0359, 1.1130, 1.1390],
          [0.8266, 0.9325, 0.9171, 0.9862, 1.0882, 1.2038],
          [0.7578, 0.8778, 0.9142, 1.0620, 1.1002, 1.2170],
          [0.7597, 0.8556, 0.9457, 1.0302, 1.1221, 1.2148],
          [0.7630, 0.8576, 0.9466, 1.0300, 1.1207, 1.2122],
          [0.7662, 0.8596, 0.9474, 1.0298, 1.1193, 1.2096],
          [0.7692, 0.8615, 0.9482, 1.0295, 1.1180, 1.2071],
          [0.7722, 0.8634, 0.9490, 1.0293, 1.1166, 1.2047],
          [0.7752, 0.8652, 0.9498, 1.0291, 1.1153, 1.2023],
          [0.7780, 0.8670, 0.9505, 1.0288, 1.1141, 1.2000],
          [0.7808, 0.8687, 0.9512, 1.0286, 1.1128, 1.1977]],
 
         [[0

#### Multi-Headed Gated Linear Self-Attention

Now that the perturbation information has been injected via cross-attention, we use self-attention to let that information propagate across all gene positions. The difference with this layer is that each position in the sequence (both context genes and target genes) can attend to every other position, allowing the model to learn how the perturbation effect spreads across the gene network.

The main differences here are that the kv_input is the sequence and we add in gating. The gating at the end to allow the model to determine how much the attention should impact each specific position.

Our gated linear attention has the following update to the calculation: 

$$\begin{aligned}
\text{elu}(x) &= \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases} \\
\\
Q &= \text{elu}(x W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(x W_k^\top + b_k) + 1.0 \\
V &= x W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q (K^\top V)}{Q (\sum_{j} K_j)^\top + \epsilon} \\
\text{Gate} &= \sigma(x W_{gate}^\top + b_{gate}) \\
\\
y &= \left( \text{Gate} \odot \text{Attn} \right) W_c^\top + b_c
\end{aligned}$$

*Note that I'll mainly call out unique calculations for self-attention and won't repeat what we've explained in cross-attention*

In [43]:
B, T_q, C = x_norm2.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 16, 6, 3)

**Self-attention** 

For self-attention, the key/value source is the same as the query source which is the normalized output from the cross-attention. By making the key/value the same, it allows each position (both context genes and query genes) to build relationships with every other position. 

*You'll notice most of the code looks the same since we built 1 attention class to handle self and cross attention.*

In [44]:
kv_input = x_norm2
kv_input

tensor([[[0.7887, 0.8571, 0.9460, 1.0424, 1.1070, 1.1984],
         [0.7747, 0.9318, 0.9806, 0.9855, 1.0874, 1.1901],
         [0.7565, 0.9252, 0.9482, 1.0422, 1.1179, 1.1562],
         [0.7910, 0.8972, 0.9849, 1.0027, 1.0793, 1.1953],
         [0.8056, 0.8475, 0.9483, 1.0049, 1.1074, 1.2237],
         [0.7707, 0.9426, 0.9531, 1.0359, 1.1130, 1.1390],
         [0.8266, 0.9325, 0.9171, 0.9862, 1.0882, 1.2038],
         [0.7578, 0.8778, 0.9142, 1.0620, 1.1002, 1.2170],
         [0.7597, 0.8556, 0.9457, 1.0302, 1.1221, 1.2148],
         [0.7630, 0.8576, 0.9466, 1.0300, 1.1207, 1.2122],
         [0.7662, 0.8596, 0.9474, 1.0298, 1.1193, 1.2096],
         [0.7692, 0.8615, 0.9482, 1.0295, 1.1180, 1.2071],
         [0.7722, 0.8634, 0.9490, 1.0293, 1.1166, 1.2047],
         [0.7752, 0.8652, 0.9498, 1.0291, 1.1153, 1.2023],
         [0.7780, 0.8670, 0.9505, 1.0288, 1.1141, 1.2000],
         [0.7808, 0.8687, 0.9512, 1.0286, 1.1128, 1.1977]],

        [[0.7799, 0.9080, 0.9680, 1.0103, 1.0846, 1.19

In [45]:
T_kv = kv_input.size(1)
T_kv

16

**Query** 

In [46]:
q_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj2.weight, -0.2)
nn.init.constant_(q_proj2.bias, 0)
q_proj2.weight

Parameter containing:
tensor([[-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000, -0.2000, -0.2000]],
       requires_grad=True)

In [47]:
q = q_proj2(x_norm2).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 16, 3]),
 tensor([[[[-1.1879, -1.1879, -1.1879],
           [-1.1900, -1.1900, -1.1900],
           [-1.1892, -1.1892, -1.1892],
           [-1.1901, -1.1901, -1.1901],
           [-1.1875, -1.1875, -1.1875],
           [-1.1909, -1.1909, -1.1909],
           [-1.1909, -1.1909, -1.1909],
           [-1.1858, -1.1858, -1.1858],
           [-1.1856, -1.1856, -1.1856],
           [-1.1860, -1.1860, -1.1860],
           [-1.1864, -1.1864, -1.1864],
           [-1.1867, -1.1867, -1.1867],
           [-1.1870, -1.1870, -1.1870],
           [-1.1874, -1.1874, -1.1874],
           [-1.1877, -1.1877, -1.1877],
           [-1.1880, -1.1880, -1.1880]],
 
          [[-1.1879, -1.1879, -1.1879],
           [-1.1900, -1.1900, -1.1900],
           [-1.1892, -1.1892, -1.1892],
           [-1.1901, -1.1901, -1.1901],
           [-1.1875, -1.1875, -1.1875],
           [-1.1909, -1.1909, -1.1909],
           [-1.1909, -1.1909, -1.1909],
           [-1.1858, -1.1858, -1.1858],
         

In [48]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 16, 3]),
 tensor([[[[0.3048, 0.3048, 0.3048],
           [0.3042, 0.3042, 0.3042],
           [0.3045, 0.3045, 0.3045],
           [0.3042, 0.3042, 0.3042],
           [0.3050, 0.3050, 0.3050],
           [0.3040, 0.3040, 0.3040],
           [0.3039, 0.3039, 0.3039],
           [0.3055, 0.3055, 0.3055],
           [0.3055, 0.3055, 0.3055],
           [0.3054, 0.3054, 0.3054],
           [0.3053, 0.3053, 0.3053],
           [0.3052, 0.3052, 0.3052],
           [0.3051, 0.3051, 0.3051],
           [0.3050, 0.3050, 0.3050],
           [0.3049, 0.3049, 0.3049],
           [0.3048, 0.3048, 0.3048]],
 
          [[0.3048, 0.3048, 0.3048],
           [0.3042, 0.3042, 0.3042],
           [0.3045, 0.3045, 0.3045],
           [0.3042, 0.3042, 0.3042],
           [0.3050, 0.3050, 0.3050],
           [0.3040, 0.3040, 0.3040],
           [0.3039, 0.3039, 0.3039],
           [0.3055, 0.3055, 0.3055],
           [0.3055, 0.3055, 0.3055],
           [0.3054, 0.3054, 0.3054],
       

**Key** 

With the key, the difference here is that instead it will work on the same data as the query.  This is the self-attention component. 

In [49]:
k_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj2.weight, -0.05)
nn.init.constant_(k_proj2.bias, 0)
k_proj2.weight

Parameter containing:
tensor([[-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500, -0.0500, -0.0500]],
       requires_grad=True)

In [50]:
k = k_proj2(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 16, 3]),
 tensor([[[[-0.2970, -0.2970, -0.2970],
           [-0.2975, -0.2975, -0.2975],
           [-0.2973, -0.2973, -0.2973],
           [-0.2975, -0.2975, -0.2975],
           [-0.2969, -0.2969, -0.2969],
           [-0.2977, -0.2977, -0.2977],
           [-0.2977, -0.2977, -0.2977],
           [-0.2965, -0.2965, -0.2965],
           [-0.2964, -0.2964, -0.2964],
           [-0.2965, -0.2965, -0.2965],
           [-0.2966, -0.2966, -0.2966],
           [-0.2967, -0.2967, -0.2967],
           [-0.2968, -0.2968, -0.2968],
           [-0.2968, -0.2968, -0.2968],
           [-0.2969, -0.2969, -0.2969],
           [-0.2970, -0.2970, -0.2970]],
 
          [[-0.2970, -0.2970, -0.2970],
           [-0.2975, -0.2975, -0.2975],
           [-0.2973, -0.2973, -0.2973],
           [-0.2975, -0.2975, -0.2975],
           [-0.2969, -0.2969, -0.2969],
           [-0.2977, -0.2977, -0.2977],
           [-0.2977, -0.2977, -0.2977],
           [-0.2965, -0.2965, -0.2965],
         

In [51]:
k = F.elu(k) + 1.0
k

tensor([[[[0.7431, 0.7431, 0.7431],
          [0.7427, 0.7427, 0.7427],
          [0.7428, 0.7428, 0.7428],
          [0.7427, 0.7427, 0.7427],
          [0.7431, 0.7431, 0.7431],
          [0.7425, 0.7425, 0.7425],
          [0.7425, 0.7425, 0.7425],
          [0.7435, 0.7435, 0.7435],
          [0.7435, 0.7435, 0.7435],
          [0.7434, 0.7434, 0.7434],
          [0.7433, 0.7433, 0.7433],
          [0.7433, 0.7433, 0.7433],
          [0.7432, 0.7432, 0.7432],
          [0.7432, 0.7432, 0.7432],
          [0.7431, 0.7431, 0.7431],
          [0.7430, 0.7430, 0.7430]],

         [[0.7431, 0.7431, 0.7431],
          [0.7427, 0.7427, 0.7427],
          [0.7428, 0.7428, 0.7428],
          [0.7427, 0.7427, 0.7427],
          [0.7431, 0.7431, 0.7431],
          [0.7425, 0.7425, 0.7425],
          [0.7425, 0.7425, 0.7425],
          [0.7435, 0.7435, 0.7435],
          [0.7435, 0.7435, 0.7435],
          [0.7434, 0.7434, 0.7434],
          [0.7433, 0.7433, 0.7433],
          [0.7433, 0.7433,

**Value** 
Same as the key, the value also works on the same data as the query.  This is also part of the self-attention component. 

In [52]:
v_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj2.weight, 0.9)
nn.init.constant_(v_proj2.bias, 0)
v_proj2.weight

Parameter containing:
tensor([[0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000]], requires_grad=True)

In [53]:
v = v_proj2(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 16, 3]),
 tensor([[[[5.3457, 5.3457, 5.3457],
           [5.3551, 5.3551, 5.3551],
           [5.3516, 5.3516, 5.3516],
           [5.3555, 5.3555, 5.3555],
           [5.3436, 5.3436, 5.3436],
           [5.3589, 5.3589, 5.3589],
           [5.3591, 5.3591, 5.3591],
           [5.3361, 5.3361, 5.3361],
           [5.3354, 5.3354, 5.3354],
           [5.3371, 5.3371, 5.3371],
           [5.3387, 5.3387, 5.3387],
           [5.3402, 5.3402, 5.3402],
           [5.3417, 5.3417, 5.3417],
           [5.3432, 5.3432, 5.3432],
           [5.3445, 5.3445, 5.3445],
           [5.3459, 5.3459, 5.3459]],
 
          [[5.3457, 5.3457, 5.3457],
           [5.3551, 5.3551, 5.3551],
           [5.3516, 5.3516, 5.3516],
           [5.3555, 5.3555, 5.3555],
           [5.3436, 5.3436, 5.3436],
           [5.3589, 5.3589, 5.3589],
           [5.3591, 5.3591, 5.3591],
           [5.3361, 5.3361, 5.3361],
           [5.3354, 5.3354, 5.3354],
           [5.3371, 5.3371, 5.3371],
       

**Normalization denominator** 

In [54]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[11.8889],
           [11.8889],
           [11.8889]],
 
          [[11.8889],
           [11.8889],
           [11.8889]]],
 
 
         [[[11.8885],
           [11.8885],
           [11.8885]],
 
          [[11.8885],
           [11.8885],
           [11.8885]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator** 

In [55]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 16, 1]),
 tensor([[[[0.0920],
           [0.0922],
           [0.0921],
           [0.0922],
           [0.0919],
           [0.0922],
           [0.0922],
           [0.0918],
           [0.0918],
           [0.0918],
           [0.0918],
           [0.0919],
           [0.0919],
           [0.0919],
           [0.0919],
           [0.0920]],
 
          [[0.0920],
           [0.0922],
           [0.0921],
           [0.0922],
           [0.0919],
           [0.0922],
           [0.0922],
           [0.0918],
           [0.0918],
           [0.0918],
           [0.0918],
           [0.0919],
           [0.0919],
           [0.0919],
           [0.0919],
           [0.0920]]],
 
 
         [[[0.0921],
           [0.0917],
           [0.0921],
           [0.0922],
           [0.0921],
           [0.0922],
           [0.0922],
           [0.0922],
           [0.0918],
           [0.0918],
           [0.0918],
           [0.0919],
           [0.0919],
           [0.0919

**Numerator** 

In [56]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[63.5550, 63.5550, 63.5550],
           [63.5550, 63.5550, 63.5550],
           [63.5550, 63.5550, 63.5550]],
 
          [[63.5550, 63.5550, 63.5550],
           [63.5550, 63.5550, 63.5550],
           [63.5550, 63.5550, 63.5550]]],
 
 
         [[[63.5601, 63.5601, 63.5601],
           [63.5601, 63.5601, 63.5601],
           [63.5601, 63.5601, 63.5601]],
 
          [[63.5601, 63.5601, 63.5601],
           [63.5601, 63.5601, 63.5601],
           [63.5601, 63.5601, 63.5601]]]], grad_fn=<UnsafeViewBackward0>))

In [57]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 16, 3]),
 tensor([[[[58.1240, 58.1240, 58.1240],
           [58.0030, 58.0030, 58.0030],
           [58.0488, 58.0488, 58.0488],
           [57.9980, 57.9980, 57.9980],
           [58.1514, 58.1514, 58.1514],
           [57.9548, 57.9548, 57.9548],
           [57.9521, 57.9521, 57.9521],
           [58.2479, 58.2479, 58.2479],
           [58.2572, 58.2572, 58.2572],
           [58.2357, 58.2357, 58.2357],
           [58.2149, 58.2149, 58.2149],
           [58.1950, 58.1950, 58.1950],
           [58.1757, 58.1757, 58.1757],
           [58.1572, 58.1572, 58.1572],
           [58.1394, 58.1394, 58.1394],
           [58.1222, 58.1222, 58.1222]],
 
          [[58.1240, 58.1241, 58.1241],
           [58.0030, 58.0030, 58.0030],
           [58.0488, 58.0488, 58.0488],
           [57.9980, 57.9980, 57.9980],
           [58.1514, 58.1514, 58.1514],
           [57.9548, 57.9548, 57.9548],
           [57.9521, 57.9521, 57.9521],
           [58.2479, 58.2479, 58.2479],
         

**Linear Attention**

In [58]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 16, 3]),
 tensor([[[[5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458]],
 
          [[5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
           [5.3458, 5.3458, 5.3458],
       

**Collapse heads** 

In [59]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 16, 6]),
 tensor([[[5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458],
          [5.3458, 5.3458, 5.3458, 5.3458, 5.3458, 5.3458]],
 
         [[5

**Gating** 

Sigmoid based attention gating is a major component we add to self-attention that we do not add to cross-attention. This gating uses the Hadamard product of the gate and the attention, which lets the model learn to suppress or pass through attention output per dimension. This allows the model to decide how much of the attention result to actually use for each dimension. We use a learned linear weight and sigmoid to pull gating values between 0 and 1, allowing the model to turn down specific values. 

In [60]:
gate = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate.weight = nn.Parameter(pattern)
gate.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [61]:
y = torch.sigmoid(gate(x_norm2)) * y
y.shape, y

(torch.Size([2, 16, 6]),
 tensor([[[3.6949, 3.9738, 4.3280, 4.9290, 5.1029, 5.1984],
          [3.6961, 3.9759, 4.3306, 4.9306, 5.1041, 5.1993],
          [3.6956, 3.9751, 4.3296, 4.9299, 5.1036, 5.1990],
          [3.6961, 3.9760, 4.3307, 4.9306, 5.1042, 5.1993],
          [3.6946, 3.9733, 4.3274, 4.9286, 5.1026, 5.1982],
          [3.6966, 3.9767, 4.3316, 4.9312, 5.1046, 5.1997],
          [3.6966, 3.9768, 4.3316, 4.9312, 5.1046, 5.1997],
          [3.6937, 3.9716, 4.3253, 4.9273, 5.1017, 5.1975],
          [3.6936, 3.9714, 4.3252, 4.9272, 5.1016, 5.1974],
          [3.6938, 3.9718, 4.3256, 4.9275, 5.1018, 5.1976],
          [3.6940, 3.9722, 4.3261, 4.9278, 5.1020, 5.1977],
          [3.6942, 3.9725, 4.3265, 4.9280, 5.1022, 5.1979],
          [3.6944, 3.9729, 4.3269, 4.9283, 5.1024, 5.1980],
          [3.6946, 3.9732, 4.3273, 4.9285, 5.1026, 5.1982],
          [3.6947, 3.9735, 4.3277, 4.9288, 5.1027, 5.1983],
          [3.6949, 3.9738, 4.3280, 4.9290, 5.1029, 5.1984]],
 
         [[3

**Cross-head final projection** 

Finally, we will now project the gated attention matrix on another final linear layer. This allows the model to learn how to combine information across the different heads. 

In [62]:
c_proj2 = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj2.weight = nn.Parameter(pattern)
c_proj2.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [63]:
x_attn = c_proj2(y)
x_attn.shape, x_attn

(torch.Size([2, 16, 6]),
 tensor([[[2.4264, 3.3522, 3.6347, 3.1352, 3.4641, 4.0747],
          [2.4273, 3.3532, 3.6358, 3.1365, 3.4655, 4.0761],
          [2.4270, 3.3528, 3.6354, 3.1360, 3.4650, 4.0756],
          [2.4274, 3.3533, 3.6359, 3.1365, 3.4655, 4.0762],
          [2.4261, 3.3519, 3.6344, 3.1350, 3.4638, 4.0743],
          [2.4277, 3.3537, 3.6363, 3.1370, 3.4660, 4.0767],
          [2.4277, 3.3537, 3.6363, 3.1370, 3.4660, 4.0767],
          [2.4254, 3.3511, 3.6335, 3.1340, 3.4628, 4.0732],
          [2.4253, 3.3510, 3.6334, 3.1339, 3.4627, 4.0731],
          [2.4255, 3.3512, 3.6336, 3.1341, 3.4629, 4.0733],
          [2.4256, 3.3514, 3.6338, 3.1343, 3.4631, 4.0736],
          [2.4258, 3.3516, 3.6340, 3.1345, 3.4633, 4.0738],
          [2.4260, 3.3517, 3.6342, 3.1347, 3.4636, 4.0740],
          [2.4261, 3.3519, 3.6344, 3.1349, 3.4638, 4.0743],
          [2.4262, 3.3520, 3.6345, 3.1351, 3.4640, 4.0745],
          [2.4264, 3.3522, 3.6347, 3.1353, 3.4642, 4.0747]],
 
         [[2

#### Attention - Residual Connection 2

We now have our gated linear self attention calculated and will use a residual connection to allow gradients to bypass the attention layer. With this you'll see how much larger the residual connection impact is on our output compared to our attention. 

In [64]:
x = x + x_attn
x.shape, x

(torch.Size([2, 16, 6]),
 tensor([[[ 62.8078,  68.9773,  76.0643,  82.9471,  88.2184,  95.8293],
          [ 62.8088,  75.9784,  80.0655,  79.9484,  88.2197,  96.8308],
          [ 61.8084,  75.9780,  78.0650,  84.9479,  91.2192,  94.8302],
          [ 63.8088,  72.9784,  80.0655,  80.9484,  87.2197,  96.8308],
          [ 64.8076,  68.9771,  77.0640,  80.9469,  89.2181,  98.8290],
          [ 61.8092,  75.9788,  77.0659,  82.9489,  89.2202,  91.8313],
          [ 66.8092,  75.9788,  75.0659,  79.9489,  88.2202,  97.8314],
          [ 60.8068,  70.9763,  74.0631,  84.9459,  88.2170,  97.8278],
          [ 59.8068,  67.9762,  75.0630,  80.9458,  88.2169,  95.8278],
          [ 60.8069,  68.9764,  76.0632,  81.9460,  89.2171,  96.8280],
          [ 61.8071,  69.9766,  77.0634,  82.9462,  90.2173,  97.8283],
          [ 62.8073,  70.9767,  78.0636,  83.9464,  91.2176,  98.8285],
          [ 63.8074,  71.9769,  79.0638,  84.9466,  92.2178,  99.8287],
          [ 64.8075,  72.9771,  80.0640

#### RMSNorm 3

We'll do another round of normalization before the SwiGLU MLP. We'll use RMSNorm again.

In [65]:
rms3= RMSNorm(embed_dim)
rms3.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [66]:
x_norm3 = rms3(x)
x_norm3.shape, x_norm3

(torch.Size([2, 16, 6]),
 tensor([[[0.7858, 0.8630, 0.9516, 1.0377, 1.1037, 1.1989],
          [0.7724, 0.9343, 0.9846, 0.9831, 1.0848, 1.1907],
          [0.7550, 0.9280, 0.9535, 1.0376, 1.1142, 1.1583],
          [0.7880, 0.9012, 0.9887, 0.9996, 1.0771, 1.1958],
          [0.8020, 0.8536, 0.9537, 1.0018, 1.1041, 1.2231],
          [0.7686, 0.9448, 0.9583, 1.0314, 1.1094, 1.1419],
          [0.8222, 0.9350, 0.9238, 0.9839, 1.0857, 1.2040],
          [0.7563, 0.8827, 0.9211, 1.0565, 1.0972, 1.2167],
          [0.7580, 0.8616, 0.9514, 1.0260, 1.1181, 1.2146],
          [0.7612, 0.8634, 0.9521, 1.0258, 1.1168, 1.2121],
          [0.7642, 0.8652, 0.9529, 1.0256, 1.1155, 1.2096],
          [0.7672, 0.8670, 0.9536, 1.0254, 1.1142, 1.2072],
          [0.7701, 0.8687, 0.9543, 1.0253, 1.1130, 1.2049],
          [0.7730, 0.8704, 0.9549, 1.0251, 1.1118, 1.2026],
          [0.7757, 0.8720, 0.9556, 1.0249, 1.1106, 1.2003],
          [0.7784, 0.8736, 0.9562, 1.0247, 1.1095, 1.1981]],
 
         [[0

#### Attention - SwiGLU 

Now that we have our attention calculated, we'll introduce our nonlinearity layers. Traditionally this was a MLP, but we replaced it with a swish-gated linear unit, or SwiGLU. SwiGLU replaces the single linear transform in a standard MLP with a gated pathway with a result as follows:

$$\begin{aligned}
\text{SiLU}(Z) &= Z \odot \sigma(Z) \\
\text{gate} &= \text{SiLU}(x W_g^\top) \\
H &= x W_u^\top \\
y &= (\text{gate} \odot H) W_d^\top
\end{aligned}$$

The Hadamard product based gating lets the network learn to selectively amplify or suppress features before the final projection, giving it more expressive power per parameter than a standard two-layer MLP with ReLU/GELU at the cost of some extra compute.

*You'll notice that SwiGLU has 3 weight matrices $W_g, W_u, W_d$, instead of the typical 2 we use in MLP. In production code, you might see helper functions that convert MLP ratios to SwiGLU ratios using 2/3 multiples to maintain the number of parameters*

In [67]:
hidden_dim = 8

**Gate** 

We'll start by initializing our gating weight $W_g$ and calculating our gate. The gate will need to scale up to our hidden dimension as it will multiply directly against our weighted input. 

In [68]:
wg = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg.weight, 0.5)
wg.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [69]:
xwg = wg(x_norm3)
xwg.shape, xwg

(torch.Size([2, 16, 8]),
 tensor([[[2.9703, 2.9703, 2.9703, 2.9703, 2.9703, 2.9703, 2.9703, 2.9703],
          [2.9749, 2.9749, 2.9749, 2.9749, 2.9749, 2.9749, 2.9749, 2.9749],
          [2.9733, 2.9733, 2.9733, 2.9733, 2.9733, 2.9733, 2.9733, 2.9733],
          [2.9752, 2.9752, 2.9752, 2.9752, 2.9752, 2.9752, 2.9752, 2.9752],
          [2.9692, 2.9692, 2.9692, 2.9692, 2.9692, 2.9692, 2.9692, 2.9692],
          [2.9771, 2.9771, 2.9771, 2.9771, 2.9771, 2.9771, 2.9771, 2.9771],
          [2.9773, 2.9773, 2.9773, 2.9773, 2.9773, 2.9773, 2.9773, 2.9773],
          [2.9653, 2.9653, 2.9653, 2.9653, 2.9653, 2.9653, 2.9653, 2.9653],
          [2.9648, 2.9648, 2.9648, 2.9648, 2.9648, 2.9648, 2.9648, 2.9648],
          [2.9657, 2.9657, 2.9657, 2.9657, 2.9657, 2.9657, 2.9657, 2.9657],
          [2.9665, 2.9665, 2.9665, 2.9665, 2.9665, 2.9665, 2.9665, 2.9665],
          [2.9673, 2.9673, 2.9673, 2.9673, 2.9673, 2.9673, 2.9673, 2.9673],
          [2.9681, 2.9681, 2.9681, 2.9681, 2.9681, 2.9681, 2.96

**SiLU Non-Linearity**

SiLU will pull our negative values closer to zero. Values above 1 will remain almost linear. When combined with the learned weights, you can quickly see how this becomes a gate. 

In [70]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 16, 8]),
 tensor([[[2.8254, 2.8254, 2.8254, 2.8254, 2.8254, 2.8254, 2.8254, 2.8254],
          [2.8304, 2.8304, 2.8304, 2.8304, 2.8304, 2.8304, 2.8304, 2.8304],
          [2.8286, 2.8286, 2.8286, 2.8286, 2.8286, 2.8286, 2.8286, 2.8286],
          [2.8308, 2.8308, 2.8308, 2.8308, 2.8308, 2.8308, 2.8308, 2.8308],
          [2.8242, 2.8242, 2.8242, 2.8242, 2.8242, 2.8242, 2.8242, 2.8242],
          [2.8328, 2.8328, 2.8328, 2.8328, 2.8328, 2.8328, 2.8328, 2.8328],
          [2.8330, 2.8330, 2.8330, 2.8330, 2.8330, 2.8330, 2.8330, 2.8330],
          [2.8199, 2.8199, 2.8199, 2.8199, 2.8199, 2.8199, 2.8199, 2.8199],
          [2.8194, 2.8194, 2.8194, 2.8194, 2.8194, 2.8194, 2.8194, 2.8194],
          [2.8204, 2.8204, 2.8204, 2.8204, 2.8204, 2.8204, 2.8204, 2.8204],
          [2.8213, 2.8213, 2.8213, 2.8213, 2.8213, 2.8213, 2.8213, 2.8213],
          [2.8222, 2.8222, 2.8222, 2.8222, 2.8222, 2.8222, 2.8222, 2.8222],
          [2.8230, 2.8230, 2.8230, 2.8230, 2.8230, 2.8230, 2.82

**Weighted input** 

Now we'll need to scale up our input to the hidden dimension. We'll use a weighted layer $W_u$ allowing the model to determine how to use the different channels to create the new dimensions

In [71]:
wu = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu.weight, -0.1)
wu.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [72]:
xwu = wu(x_norm3)
xwu.shape, xwu

(torch.Size([2, 16, 8]),
 tensor([[[-0.5941, -0.5941, -0.5941, -0.5941, -0.5941, -0.5941, -0.5941,
           -0.5941],
          [-0.5950, -0.5950, -0.5950, -0.5950, -0.5950, -0.5950, -0.5950,
           -0.5950],
          [-0.5947, -0.5947, -0.5947, -0.5947, -0.5947, -0.5947, -0.5947,
           -0.5947],
          [-0.5950, -0.5950, -0.5950, -0.5950, -0.5950, -0.5950, -0.5950,
           -0.5950],
          [-0.5938, -0.5938, -0.5938, -0.5938, -0.5938, -0.5938, -0.5938,
           -0.5938],
          [-0.5954, -0.5954, -0.5954, -0.5954, -0.5954, -0.5954, -0.5954,
           -0.5954],
          [-0.5955, -0.5955, -0.5955, -0.5955, -0.5955, -0.5955, -0.5955,
           -0.5955],
          [-0.5931, -0.5931, -0.5931, -0.5931, -0.5931, -0.5931, -0.5931,
           -0.5931],
          [-0.5930, -0.5930, -0.5930, -0.5930, -0.5930, -0.5930, -0.5930,
           -0.5930],
          [-0.5931, -0.5931, -0.5931, -0.5931, -0.5931, -0.5931, -0.5931,
           -0.5931],
          [-0.5933, -0.59

**Apply Gate** 

Now we'll go ahead and apply the gate. We take the Hadamard product which allows the model to gate each value of the scaled up projection. 

In [73]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 16, 8]),
 tensor([[[-1.6785, -1.6785, -1.6785, -1.6785, -1.6785, -1.6785, -1.6785,
           -1.6785],
          [-1.6841, -1.6841, -1.6841, -1.6841, -1.6841, -1.6841, -1.6841,
           -1.6841],
          [-1.6821, -1.6821, -1.6821, -1.6821, -1.6821, -1.6821, -1.6821,
           -1.6821],
          [-1.6844, -1.6844, -1.6844, -1.6844, -1.6844, -1.6844, -1.6844,
           -1.6844],
          [-1.6771, -1.6771, -1.6771, -1.6771, -1.6771, -1.6771, -1.6771,
           -1.6771],
          [-1.6868, -1.6868, -1.6868, -1.6868, -1.6868, -1.6868, -1.6868,
           -1.6868],
          [-1.6869, -1.6869, -1.6869, -1.6869, -1.6869, -1.6869, -1.6869,
           -1.6869],
          [-1.6723, -1.6723, -1.6723, -1.6723, -1.6723, -1.6723, -1.6723,
           -1.6723],
          [-1.6718, -1.6718, -1.6718, -1.6718, -1.6718, -1.6718, -1.6718,
           -1.6718],
          [-1.6729, -1.6729, -1.6729, -1.6729, -1.6729, -1.6729, -1.6729,
           -1.6729],
          [-1.6739, -1.67

**Project Down**

Now we need to project back down to our embedding dimension. We'll use a final weighted $W_d$ layer to determine how to project back down. This is similar to the final layer of an MLP. 

In [74]:
wd = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd.weight, 0.33)
wd.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [75]:
xswig = wd(xw)
xswig.shape, xswig

(torch.Size([2, 16, 6]),
 tensor([[[-4.4312, -4.4312, -4.4312, -4.4312, -4.4312, -4.4312],
          [-4.4460, -4.4460, -4.4460, -4.4460, -4.4460, -4.4460],
          [-4.4406, -4.4406, -4.4406, -4.4406, -4.4406, -4.4406],
          [-4.4469, -4.4469, -4.4469, -4.4469, -4.4469, -4.4469],
          [-4.4275, -4.4275, -4.4275, -4.4275, -4.4275, -4.4275],
          [-4.4530, -4.4530, -4.4530, -4.4530, -4.4530, -4.4530],
          [-4.4535, -4.4535, -4.4535, -4.4535, -4.4535, -4.4535],
          [-4.4150, -4.4150, -4.4150, -4.4150, -4.4150, -4.4150],
          [-4.4136, -4.4136, -4.4136, -4.4136, -4.4136, -4.4136],
          [-4.4164, -4.4164, -4.4164, -4.4164, -4.4164, -4.4164],
          [-4.4190, -4.4190, -4.4190, -4.4190, -4.4190, -4.4190],
          [-4.4216, -4.4216, -4.4216, -4.4216, -4.4216, -4.4216],
          [-4.4241, -4.4241, -4.4241, -4.4241, -4.4241, -4.4241],
          [-4.4265, -4.4265, -4.4265, -4.4265, -4.4265, -4.4265],
          [-4.4288, -4.4288, -4.4288, -4.4288, -4.4

#### Residual Connection 3

We now have our SwiGLU based projection calculated and will use a residual connection to allow gradients to bypass the SwiGLU calculations. With this you'll see how much larger the residual connection impact is on our output compared to our SwiGLU output. 

In [76]:
x = x + xswig
x.shape, x

(torch.Size([2, 16, 6]),
 tensor([[[58.3767, 64.5462, 71.6331, 78.5160, 83.7872, 91.3982],
          [58.3628, 71.5324, 75.6195, 75.5024, 83.7737, 92.3848],
          [57.3678, 71.5374, 73.6244, 80.5073, 86.7785, 90.3896],
          [59.3619, 68.5316, 75.6186, 76.5015, 82.7729, 92.3839],
          [60.3801, 64.5496, 72.6366, 76.5194, 84.7906, 94.4015],
          [57.3561, 71.5258, 72.6129, 78.4959, 84.7672, 87.3783],
          [62.3557, 71.5253, 70.6124, 75.4954, 83.7667, 93.3779],
          [56.3918, 66.5613, 69.6481, 80.5309, 83.8020, 93.4128],
          [55.3932, 63.5626, 70.6495, 76.5322, 83.8033, 91.4142],
          [56.3906, 64.5600, 71.6469, 77.5296, 84.8008, 92.4116],
          [57.3881, 65.5575, 72.6444, 78.5272, 85.7983, 93.4092],
          [58.3856, 66.5551, 73.6420, 79.5248, 86.7959, 94.4069],
          [59.3833, 67.5528, 74.6397, 80.5225, 87.7937, 95.4046],
          [60.3810, 68.5505, 75.6375, 81.5203, 88.7915, 96.4024],
          [61.3789, 69.5484, 76.6353, 82.5181, 89.7

### Final Layer Normalization

The previous layers can run sequentially for as many transformer layers as configured. The more layers, the "deeper" the network becomes and the more parameters/capacity the model has (not always good). Once all the layers have completed, we're ready for a final normalization. Like previous normalizations this will pull our values together to focus on the variance. This normalized output is what we'll use to extract our predictions.

In [77]:
rmsf = RMSNorm(embed_dim)
rmsf.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [78]:
sequence = rmsf(x)
sequence.shape, sequence

(torch.Size([2, 16, 6]),
 tensor([[[0.7727, 0.8544, 0.9482, 1.0393, 1.1091, 1.2098],
          [0.7588, 0.9300, 0.9832, 0.9816, 1.0892, 1.2011],
          [0.7405, 0.9234, 0.9503, 1.0392, 1.1201, 1.1667],
          [0.7753, 0.8950, 0.9876, 0.9991, 1.0810, 1.2065],
          [0.7901, 0.8446, 0.9504, 1.0012, 1.1095, 1.2352],
          [0.7546, 0.9411, 0.9554, 1.0328, 1.1153, 1.1496],
          [0.8115, 0.9308, 0.9190, 0.9825, 1.0901, 1.2152],
          [0.7416, 0.8753, 0.9159, 1.0590, 1.1020, 1.2284],
          [0.7431, 0.8527, 0.9478, 1.0267, 1.1243, 1.2264],
          [0.7467, 0.8548, 0.9487, 1.0266, 1.1228, 1.2236],
          [0.7501, 0.8569, 0.9495, 1.0264, 1.1214, 1.2209],
          [0.7534, 0.8588, 0.9503, 1.0262, 1.1200, 1.2182],
          [0.7567, 0.8608, 0.9511, 1.0260, 1.1187, 1.2157],
          [0.7598, 0.8626, 0.9518, 1.0259, 1.1174, 1.2131],
          [0.7629, 0.8645, 0.9526, 1.0257, 1.1161, 1.2107],
          [0.7659, 0.8662, 0.9533, 1.0255, 1.1148, 1.2082]],
 
         [[0

### Extract Predicted Gene Representations

Now we extract only the query positions from the sequence. Recall that we concatenated `[context_latents, target_indices]` to create our sequence `x`, so the second half contains the predictions for each gene's post-perturbation state. 

In [79]:
predictions = sequence[:, num_genes:, :]
predictions.shape, predictions

(torch.Size([2, 8, 6]),
 tensor([[[0.7431, 0.8527, 0.9478, 1.0267, 1.1243, 1.2264],
          [0.7467, 0.8548, 0.9487, 1.0266, 1.1228, 1.2236],
          [0.7501, 0.8569, 0.9495, 1.0264, 1.1214, 1.2209],
          [0.7534, 0.8588, 0.9503, 1.0262, 1.1200, 1.2182],
          [0.7567, 0.8608, 0.9511, 1.0260, 1.1187, 1.2157],
          [0.7598, 0.8626, 0.9518, 1.0259, 1.1174, 1.2131],
          [0.7629, 0.8645, 0.9526, 1.0257, 1.1161, 1.2107],
          [0.7659, 0.8662, 0.9533, 1.0255, 1.1148, 1.2082]],
 
         [[0.7431, 0.8527, 0.9478, 1.0267, 1.1243, 1.2264],
          [0.7467, 0.8548, 0.9487, 1.0266, 1.1228, 1.2236],
          [0.7501, 0.8569, 0.9495, 1.0264, 1.1214, 1.2209],
          [0.7534, 0.8588, 0.9503, 1.0262, 1.1200, 1.2182],
          [0.7567, 0.8608, 0.9511, 1.0260, 1.1187, 1.2157],
          [0.7598, 0.8626, 0.9518, 1.0259, 1.1174, 1.2131],
          [0.7629, 0.8645, 0.9526, 1.0257, 1.1161, 1.2107],
          [0.7659, 0.8662, 0.9533, 1.0255, 1.1148, 1.2082]]],
        gra

### Mean Prediction Head $\mu$

The mean head projects each gene's predicted mean representation into the final latent space. This gives us the average (mean) predicted latent representation for each gene after the perturbation. 

In [80]:
head_mu = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(d).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1.0*(rows + 1*cols)  
head_mu.weight = nn.Parameter(pattern)
nn.init.zeros_(head_mu.bias)
head_mu.weight, head_mu.bias

(Parameter containing:
 tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
         [ 1.,  2.,  3.,  4.,  5.,  6.],
         [ 2.,  3.,  4.,  5.,  6.,  7.],
         [ 3.,  4.,  5.,  6.,  7.,  8.],
         [ 4.,  5.,  6.,  7.,  8.,  9.],
         [ 5.,  6.,  7.,  8.,  9., 10.]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [81]:
mu = head_mu(predictions)
mu.shape, mu

(torch.Size([2, 8, 6]),
 tensor([[[16.4576, 22.3787, 28.2999, 34.2210, 40.1421, 46.0632],
          [16.4412, 22.3644, 28.2875, 34.2107, 40.1338, 46.0570],
          [16.4251, 22.3503, 28.2754, 34.2005, 40.1257, 46.0508],
          [16.4094, 22.3364, 28.2635, 34.1905, 40.1176, 46.0446],
          [16.3940, 22.3229, 28.2517, 34.1806, 40.1095, 46.0383],
          [16.3789, 22.3095, 28.2402, 34.1708, 40.1014, 46.0321],
          [16.3641, 22.2965, 28.2288, 34.1611, 40.0935, 46.0258],
          [16.3496, 22.2836, 28.2176, 34.1516, 40.0855, 46.0195]],
 
         [[16.4576, 22.3787, 28.2998, 34.2209, 40.1420, 46.0631],
          [16.4412, 22.3644, 28.2875, 34.2107, 40.1338, 46.0570],
          [16.4251, 22.3503, 28.2754, 34.2005, 40.1257, 46.0508],
          [16.4094, 22.3364, 28.2635, 34.1905, 40.1176, 46.0446],
          [16.3940, 22.3229, 28.2517, 34.1806, 40.1095, 46.0383],
          [16.3789, 22.3095, 28.2402, 34.1708, 40.1014, 46.0321],
          [16.3641, 22.2965, 28.2288, 34.1611, 40

### Log-Variance Prediction Head $\log\sigma^2$

The log-variance head projects each gene's prediction into an uncertainty estimate. Higher values indicate the model is less confident in its prediction for that gene. We predict log-variance rather than variance directly because the log scale is numerically stable, unconstrained and closer together so there's less incentive to get only large values correct. 

In [82]:
head_logvar = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(d).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 0.01*(rows - 0.1*cols)  
head_logvar.weight = nn.Parameter(pattern)
nn.init.zeros_(head_logvar.bias)
head_logvar.weight, head_logvar.bias

(Parameter containing:
 tensor([[ 0.0000,  0.0100,  0.0200,  0.0300,  0.0400,  0.0500],
         [-0.0010,  0.0090,  0.0190,  0.0290,  0.0390,  0.0490],
         [-0.0020,  0.0080,  0.0180,  0.0280,  0.0380,  0.0480],
         [-0.0030,  0.0070,  0.0170,  0.0270,  0.0370,  0.0470],
         [-0.0040,  0.0060,  0.0160,  0.0260,  0.0360,  0.0460],
         [-0.0050,  0.0050,  0.0150,  0.0250,  0.0350,  0.0450]],
        requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [83]:
logvar = head_logvar(predictions.float())
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]],
 
         [[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]]],
        gra

**Clamp logvar**

We bind the predicted variance to a sane range. Since $\text{var} = e^{\text{logvar}}$:

- $\text{logvar} = -10 \implies \text{var} \approx 0.00005$ (very confident, near-deterministic)
- $\text{logvar} = 2 \implies \text{var} \approx 7.4$ (high uncertainty)

Without clamping, the model could predict extreme logvar values early in training. Very negative values cause the Gaussian NLL loss to divide by near-zero variance (exploding loss), while very positive values let the model get away with poor predictions by saying "I'm maximally uncertain" (loss collapse). The clamp keeps the loss well-behaved from the start.

In [84]:
logvar = torch.clamp(logvar, min=-10, max=2)
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]],
 
         [[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]]],
        gra

## AC Predictor Complete

We now have for the perturbed cell, each gene's latent dimension mean ($\mu$) and log-variance ($\log\sigma^2$) predictions. You'll see that both outputs have shape $[B, \text{num\_genes}, \text{embed\_dim}]$ one latent vector per gene per sample.

In [85]:
mu.shape, mu

(torch.Size([2, 8, 6]),
 tensor([[[16.4576, 22.3787, 28.2999, 34.2210, 40.1421, 46.0632],
          [16.4412, 22.3644, 28.2875, 34.2107, 40.1338, 46.0570],
          [16.4251, 22.3503, 28.2754, 34.2005, 40.1257, 46.0508],
          [16.4094, 22.3364, 28.2635, 34.1905, 40.1176, 46.0446],
          [16.3940, 22.3229, 28.2517, 34.1806, 40.1095, 46.0383],
          [16.3789, 22.3095, 28.2402, 34.1708, 40.1014, 46.0321],
          [16.3641, 22.2965, 28.2288, 34.1611, 40.0935, 46.0258],
          [16.3496, 22.2836, 28.2176, 34.1516, 40.0855, 46.0195]],
 
         [[16.4576, 22.3787, 28.2998, 34.2209, 40.1420, 46.0631],
          [16.4412, 22.3644, 28.2875, 34.2107, 40.1338, 46.0570],
          [16.4251, 22.3503, 28.2754, 34.2005, 40.1257, 46.0508],
          [16.4094, 22.3364, 28.2635, 34.1905, 40.1176, 46.0446],
          [16.3940, 22.3229, 28.2517, 34.1806, 40.1095, 46.0383],
          [16.3789, 22.3095, 28.2402, 34.1708, 40.1014, 46.0321],
          [16.3641, 22.2965, 28.2288, 34.1611, 40

In [86]:
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]],
 
         [[0.1646, 0.1587, 0.1527, 0.1468, 0.1409, 0.1350],
          [0.1644, 0.1585, 0.1526, 0.1466, 0.1407, 0.1348],
          [0.1643, 0.1583, 0.1524, 0.1465, 0.1406, 0.1346],
          [0.1641, 0.1582, 0.1522, 0.1463, 0.1404, 0.1345],
          [0.1639, 0.1580, 0.1521, 0.1462, 0.1402, 0.1343],
          [0.1638, 0.1579, 0.1519, 0.1460, 0.1401, 0.1341],
          [0.1636, 0.1577, 0.1518, 0.1458, 0.1399, 0.1340],
          [0.1635, 0.1576, 0.1516, 0.1457, 0.1398, 0.1338]]],
        gra

# Latent Representation

We now have a latent representation of our predicted perturbed cell states. You'll notice that the shape still contains the batch, our context length (number of genes), and the embedding dimensions. The ACPredictor works within this latent space to predict a mean latent representation and an uncertainty estimate. During training we calculate the Gaussian NLL loss and VICReg loss of these values against the expected representation from the teacher encoder's target cell representation.